# CARec encoder and dimension study

- Load the saved Amazon Electronics catalogue, interaction partitions, and ID maps.
- Use the `NI-PML benchmarks` kernel from the project `.venv`.
- Data exploration, text preparation, sampling, splitting, and saving belong in [data_exploration.ipynb](data_exploration.ipynb).


In [24]:
from pathlib import Path
import gzip
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
PREPARED_DIR = PROJECT_ROOT / "data/processed/electronics_carec_035_seed2023"
print(f"Prepared dataset: {PREPARED_DIR}")


Prepared dataset: /Users/matouskovar/FIT/NI-PML/data/processed/electronics_carec_035_seed2023


## Load products and ID maps

- Read one product record per JSONL line.
- Load user and product ID maps, keeping their IDs as strings.


In [25]:
items_path = PREPARED_DIR / "items.jsonl.gz"
product_records = []

with gzip.open(items_path, "rt", encoding="utf-8") as source:
    for line in source:
        product = json.loads(line)
        product_records.append(product)

items = pd.DataFrame(product_records)
del product_records

user_map_path = PREPARED_DIR / "user_ids.tsv.gz"
user_map = pd.read_csv(user_map_path, sep="\t", dtype={"user_id": str})

item_map_path = PREPARED_DIR / "item_ids.tsv.gz"
item_map = pd.read_csv(item_map_path, sep="\t", dtype={"item_id": str})


## Load interaction partitions

- Load train, validation, test, and cold interactions.
- Keep user and product IDs as strings and preserve float precision.


In [26]:
partition_names = ("train", "valid", "test", "cold")

interaction_types = {
    "user_id:token": str,
    "item_id:token": str,
}
partitions = {}

for name in partition_names:
    partition_path = PREPARED_DIR / f"{name}.tsv.gz"
    frame = pd.read_csv(
        partition_path,
        sep="\t",
        dtype=interaction_types,
        float_precision="round_trip",
    )
    partitions[name] = frame

train = partitions["train"]
valid = partitions["valid"]
test = partitions["test"]
cold = partitions["cold"]


## Show the loaded data

- Print product, user, and partition counts.
- Show two product texts.


In [27]:
print(f"Loaded {len(items):,} products and {len(user_map):,} users")

for name, frame in partitions.items():
    print(f"{name}: {len(frame):,} interactions")

text_preview = items[["item_id:token", "text"]].head(2)
display(text_preview)


Loaded 34,220 products and 81,520 users
train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions


,item_id:token,text
0,0545105668,Elephant Run Visit Amazon's Roland Smith Page ...
1,0789743035,My MacBook Visit Amazon's John Ray Page Compu...


## Encoder candidates

- `text` is the saved product text from `items["text"]`; `instruction` describes the recommendation task. Its wording is still to be chosen.
- Run the three instruction models with and without instructions: plain `text` for Qwen/E5-instruct, `["", text]` for INSTRUCTOR. Pilot these variants before bulk generation.
- E5-base's `query: ` prefix is required formatting, not a task instruction. `\n` below means a newline.

| Model | Instruction-based | Input format |
| --- | --- | --- |
| [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | No | `text` |
| [all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | No | `text` |
| [e5-base-v2](https://huggingface.co/intfloat/e5-base-v2) | No | `query: {text}` |
| [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | Yes | `Instruct: {instruction}\nQuery:{text}` |
| [multilingual-e5-large-instruct](https://huggingface.co/intfloat/multilingual-e5-large-instruct) | Yes | `Instruct: {instruction}\nQuery: {text}` |
| [instructor-base](https://huggingface.co/hkunlp/instructor-base) | Yes | `[instruction, text]` via `InstructorEmbedding` |
